In [6]:
# ============================================================
# NOTEBOOK 03: SEMI-SUPERVISED MODEL TRAINING
# ImmuniWatch Nigeria | AIRS Production System
# ============================================================

!pip install transformers datasets huggingface_hub scikit-learn -q


In [7]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from sklearn.metrics import classification_report, f1_score
from huggingface_hub import hf_hub_download
import warnings

warnings.filterwarnings('ignore')

# Verify GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Device: cuda
GPU: Tesla T4
Device: cuda
GPU: Tesla T4


In [8]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully")

Mounted at /content/drive
Google Drive mounted successfully


In [9]:
# ============================================================
# CENTRAL CONFIGURATION
# ============================================================

CONFIG = {
    # Model
    'model_name': 'xlm-roberta-base',
    'num_labels': 3,
    'max_length': 128,

    # Training
    'batch_size': 16,
    'num_epochs': 5,
    'learning_rate': 2e-5,
    'warmup_ratio': 0.1,
    'weight_decay': 0.01,

    # Semi-supervised
    'pseudo_label_threshold': 0.85,
    'pseudo_label_epochs': 2,

    # Paths
    'output_dir': 'immuniwatch_model',
    'repo_id': 'AHFIDAILabs/immuniwatch',

    # Label mapping
    'label2id': {'Misinformation': 0, 'Factual': 1, 'Uncertain': 2},
    'id2label': {0: 'Misinformation', 1: 'Factual', 2: 'Uncertain'},

    # Class weights from notebook 02
    'class_weights': [5.4243, 0.8664, 0.6019]
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
print("Configuration set")
print(f"Label mapping: {CONFIG['label2id']}")
print(f"Class weights: {CONFIG['class_weights']}")

Configuration set
Label mapping: {'Misinformation': 0, 'Factual': 1, 'Uncertain': 2}
Class weights: [5.4243, 0.8664, 0.6019]


In [10]:
# Load train split
train_file = hf_hub_download(
    repo_id=CONFIG['repo_id'],
    filename='train.csv',
    repo_type='dataset'
)
train_df = pd.read_csv(train_file)

# Load val split
val_file = hf_hub_download(
    repo_id=CONFIG['repo_id'],
    filename='val.csv',
    repo_type='dataset'
)
val_df = pd.read_csv(val_file)

# Load unlabeled data for semi-supervised learning
unlabeled_file = hf_hub_download(
    repo_id=CONFIG['repo_id'],
    filename='unlabeled_tweets_for_human_review.csv',
    repo_type='dataset'
)
unlabeled_df = pd.read_csv(unlabeled_file)

print(f"Train:     {len(train_df):,} rows")
print(f"Val:       {len(val_df):,} rows")
print(f"Unlabeled: {len(unlabeled_df):,} rows")

train.csv: 0.00B [00:00, ?B/s]

val.csv: 0.00B [00:00, ?B/s]

unlabeled_tweets_for_human_review.csv: 0.00B [00:00, ?B/s]

Train:     8,169 rows
Val:       1,751 rows
Unlabeled: 33,334 rows


In [11]:
#processing

# Standardize labels
for df in [train_df, val_df]:
    df['predicted_label'] = df['predicted_label'].str.strip().str.capitalize()

# Encode labels
train_df['label_id'] = train_df['predicted_label'].map(CONFIG['label2id'])
val_df['label_id'] = val_df['predicted_label'].map(CONFIG['label2id'])

# Use clean_Text column, fall back to Tweet_Text if needed
text_col = 'clean_Text' if 'clean_Text' in train_df.columns else 'Tweet_Text'
print(f"Using text column: {text_col}")

# Fill any null text
train_df[text_col] = train_df[text_col].fillna('').astype(str)
val_df[text_col] = val_df[text_col].fillna('').astype(str)
unlabeled_df[text_col] = unlabeled_df[text_col].fillna('').astype(str)

print(f"Train label distribution:")
print(train_df['predicted_label'].value_counts())
print(f"\nVal label distribution:")
print(val_df['predicted_label'].value_counts())

Using text column: Tweet_Text
Train label distribution:
predicted_label
Uncertain         4524
Factual           3143
Misinformation     502
Name: count, dtype: int64

Val label distribution:
predicted_label
Uncertain         970
Factual           673
Misinformation    108
Name: count, dtype: int64


In [12]:
#LOADING TOKENIZER

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

class TweetDataset(Dataset):
    def __init__(self, texts, labels=None, max_length=128):
        self.texts = texts
        self.labels = labels
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze()
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print("Tokenizer loaded and Dataset class defined")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded and Dataset class defined


In [13]:
print("=== CREATING DATALOADERS ===")

train_dataset = TweetDataset(
    texts=train_df[text_col].tolist(),
    labels=train_df['label_id'].tolist(),
    max_length=CONFIG['max_length']
)

val_dataset = TweetDataset(
    texts=val_df[text_col].tolist(),
    labels=val_df['label_id'].tolist(),
    max_length=CONFIG['max_length']
)

unlabeled_dataset = TweetDataset(
    texts=unlabeled_df[text_col].tolist(),
    labels=None,
    max_length=CONFIG['max_length']
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False
)

unlabeled_loader = DataLoader(
    unlabeled_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False
)

print(f"Train batches:     {len(train_loader)}")
print(f"Val batches:       {len(val_loader)}")
print(f"Unlabeled batches: {len(unlabeled_loader)}")

=== CREATING DATALOADERS ===
Train batches:     511
Val batches:       110
Unlabeled batches: 2084


In [14]:
#LOADING XLM-ROBERTA MODEL

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG['model_name'],
    num_labels=CONFIG['num_labels'],
    id2label=CONFIG['id2label'],
    label2id=CONFIG['label2id']
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model loaded: {CONFIG['model_name']}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: xlm-roberta-base
Total parameters:     278,045,955
Trainable parameters: 278,045,955


In [15]:
# ============================================================
# COST-SENSITIVE LEARNING
# Higher weight on Misinformation class (5.4243)
# ============================================================

class_weights_tensor = torch.tensor(
    CONFIG['class_weights'],
    dtype=torch.float
).to(device)

loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

print("Cost-sensitive loss function defined")
print(f"Class weights applied:")
for label, weight in zip(CONFIG['id2label'].values(), CONFIG['class_weights']):
    print(f"  {label:<20} {weight}")

Cost-sensitive loss function defined
Class weights applied:
  Misinformation       5.4243
  Factual              0.8664
  Uncertain            0.6019


In [16]:
total_steps = len(train_loader) * CONFIG['num_epochs']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])

optimizer = AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Optimizer: AdamW")
print(f"Learning rate:  {CONFIG['learning_rate']}")
print(f"Total steps:    {total_steps}")
print(f"Warmup steps:   {warmup_steps}")

Optimizer: AdamW
Learning rate:  2e-05
Total steps:    2555
Warmup steps:   255


In [17]:
def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_labels, all_preds, average='macro')

    return avg_loss, f1, all_preds, all_labels


In [18]:
# ============================================================
# PHASE 1: SUPERVISED FINE-TUNING ON LABELED DATA
# ============================================================

print("=== PHASE 1: SUPERVISED FINE-TUNING ===")
print(f"Epochs: {CONFIG['num_epochs']}")
print(f"Training on {len(train_df):,} labeled tweets")

best_val_f1 = 0
best_model_state = None
training_history = []

for epoch in range(CONFIG['num_epochs']):
    model.train()
    total_train_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_fn(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        if (batch_idx + 1) % 50 == 0:
            print(f"  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} "
                  f"| Loss: {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_loader)
    val_loss, val_f1, _, _ = evaluate(model, val_loader, device)

    training_history.append({
        'epoch': epoch + 1,
        'train_loss': round(avg_train_loss, 4),
        'val_loss': round(val_loss, 4),
        'val_f1': round(val_f1, 4)
    })

    print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val F1:     {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"  Best model saved (Val F1: {best_val_f1:.4f})")

print(f"\nPhase 1 complete. Best Val F1: {best_val_f1:.4f}")

=== PHASE 1: SUPERVISED FINE-TUNING ===
Epochs: 5
Training on 8,169 labeled tweets
  Epoch 1 | Batch 50/511 | Loss: 1.1050
  Epoch 1 | Batch 100/511 | Loss: 1.1462
  Epoch 1 | Batch 150/511 | Loss: 0.9563
  Epoch 1 | Batch 200/511 | Loss: 1.2219
  Epoch 1 | Batch 250/511 | Loss: 0.8894
  Epoch 1 | Batch 300/511 | Loss: 0.6751
  Epoch 1 | Batch 350/511 | Loss: 0.4040
  Epoch 1 | Batch 400/511 | Loss: 1.0008
  Epoch 1 | Batch 450/511 | Loss: 0.5360
  Epoch 1 | Batch 500/511 | Loss: 0.6064

Epoch 1/5
  Train Loss: 0.9311
  Val Loss:   0.6479
  Val F1:     0.7332
  Best model saved (Val F1: 0.7332)
  Epoch 2 | Batch 50/511 | Loss: 0.3358
  Epoch 2 | Batch 100/511 | Loss: 1.1199
  Epoch 2 | Batch 150/511 | Loss: 0.2332
  Epoch 2 | Batch 200/511 | Loss: 0.2781
  Epoch 2 | Batch 250/511 | Loss: 0.1918
  Epoch 2 | Batch 300/511 | Loss: 0.8713
  Epoch 2 | Batch 350/511 | Loss: 0.3657
  Epoch 2 | Batch 400/511 | Loss: 0.1095
  Epoch 2 | Batch 450/511 | Loss: 0.0337
  Epoch 2 | Batch 500/511 | Lo

In [ ]:
# ============================================================
# PHASE 2: SEMI-SUPERVISED - GENERATE PSEUDO LABELS AND RETRAIN
# ============================================================


print("=== PHASE 2: GENERATING PSEUDO LABELS ===")
print(f"Confidence threshold: {CONFIG['pseudo_label_threshold']}")

model.load_state_dict(best_model_state)
model.eval()

all_pseudo_texts = []
all_pseudo_labels = []

with torch.no_grad():
    for batch_start in range(0, len(unlabeled_df), CONFIG['batch_size']):
        batch_texts = unlabeled_df[text_col].iloc[
            batch_start:batch_start + CONFIG['batch_size']
        ].tolist()

        encoding = tokenizer(
            batch_texts,
            max_length=CONFIG['max_length'],
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=1)
        confidences = torch.max(probs, dim=1).values
        preds = torch.argmax(probs, dim=1)

        for i in range(len(batch_texts)):
            if confidences[i].item() >= CONFIG['pseudo_label_threshold']:
                all_pseudo_texts.append(batch_texts[i])
                all_pseudo_labels.append(preds[i].item())

print(f"Total unlabeled tweets:        {len(unlabeled_df):,}")
print(f"High-confidence pseudo labels: {len(all_pseudo_texts):,}")
print(f"Below threshold (discarded):   {len(unlabeled_df) - len(all_pseudo_texts):,}")

if len(all_pseudo_labels) > 0:
    from collections import Counter
    dist = Counter(all_pseudo_labels)
    for label_id, count in sorted(dist.items()):
        print(f"  {CONFIG['id2label'][label_id]}: {count:,}")

    # Combine labeled train + pseudo labeled
    combined_texts = train_df[text_col].tolist() + all_pseudo_texts
    combined_labels = train_df['label_id'].tolist() + all_pseudo_labels

    print(f"\nOriginal labeled:  {len(train_df):,}")
    print(f"Pseudo labeled:    {len(all_pseudo_texts):,}")
    print(f"Combined total:    {len(combined_texts):,}")

    combined_dataset = TweetDataset(
        texts=combined_texts,
        labels=combined_labels,
        max_length=CONFIG['max_length']
    )

    combined_loader = DataLoader(
        combined_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=True
    )

    total_steps_p2 = len(combined_loader) * CONFIG['pseudo_label_epochs']
    warmup_steps_p2 = int(total_steps_p2 * CONFIG['warmup_ratio'])

    optimizer_p2 = AdamW(
        model.parameters(),
        lr=CONFIG['learning_rate'] / 2,
        weight_decay=CONFIG['weight_decay']
    )

    scheduler_p2 = get_linear_schedule_with_warmup(
        optimizer_p2,
        num_warmup_steps=warmup_steps_p2,
        num_training_steps=total_steps_p2
    )

    for epoch in range(CONFIG['pseudo_label_epochs']):
        model.train()
        total_loss = 0

        for batch_idx, batch in enumerate(combined_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer_p2.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer_p2.step()
            scheduler_p2.step()

            total_loss += loss.item()

            if (batch_idx + 1) % 100 == 0:
                print(f"  Epoch {epoch+1} | Batch {batch_idx+1} "
                      f"| Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(combined_loader)
        val_loss, val_f1, _, _ = evaluate(model, val_loader, device)

        print(f"\nPhase 2 Epoch {epoch+1}/{CONFIG['pseudo_label_epochs']}")
        print(f"  Train Loss: {avg_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        print(f"  Val F1:     {val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = {
                k: v.clone() for k, v in model.state_dict().items()
            }
            print(f"  Best model updated (Val F1: {best_val_f1:.4f})")

    print(f"\nPhase 2 complete. Best Val F1: {best_val_f1:.4f}")

else:
    print("No pseudo labels above threshold. Proceeding with phase 1 best model.")

=== PHASE 2: GENERATING PSEUDO LABELS ===
Confidence threshold: 0.85
Total unlabeled tweets:        33,334
High-confidence pseudo labels: 31,859
Below threshold (discarded):   1,475
  Misinformation: 336
  Factual: 5,935
  Uncertain: 25,588

Original labeled:  8,169
Pseudo labeled:    31,859
Combined total:    40,028
  Epoch 1 | Batch 100 | Loss: 0.0586
  Epoch 1 | Batch 200 | Loss: 0.2187
  Epoch 1 | Batch 300 | Loss: 0.0329
  Epoch 1 | Batch 400 | Loss: 3.1590
  Epoch 1 | Batch 500 | Loss: 0.4505
  Epoch 1 | Batch 600 | Loss: 0.2023
  Epoch 1 | Batch 700 | Loss: 0.0743
  Epoch 1 | Batch 800 | Loss: 0.0016
  Epoch 1 | Batch 900 | Loss: 0.0018
  Epoch 1 | Batch 1000 | Loss: 0.0137
  Epoch 1 | Batch 1100 | Loss: 0.3122
  Epoch 1 | Batch 1200 | Loss: 0.1270
  Epoch 1 | Batch 1300 | Loss: 0.2458
  Epoch 1 | Batch 1400 | Loss: 0.3789
  Epoch 1 | Batch 1500 | Loss: 0.0031
  Epoch 1 | Batch 1600 | Loss: 0.3516
  Epoch 1 | Batch 1700 | Loss: 0.0008
  Epoch 1 | Batch 1800 | Loss: 0.0007
  Epoc

In [25]:
#print("=== LOADING BEST MODEL FOR FINAL EVALUATION ===")

model.load_state_dict(best_model_state)

val_loss, val_f1, val_preds, val_labels = evaluate(model, val_loader, device)

print(f"Final Val Loss: {val_loss:.4f}")
print(f"Final Val F1:   {val_f1:.4f}")
print(f"\nDetailed Classification Report:")
print(classification_report(
    val_labels,
    val_preds,
    target_names=list(CONFIG['id2label'].values())
))

Final Val Loss: 0.3242
Final Val F1:   0.9413

Detailed Classification Report:
                precision    recall  f1-score   support

Misinformation       0.84      0.91      0.87       108
       Factual       0.97      0.99      0.98       673
     Uncertain       0.99      0.97      0.98       970

      accuracy                           0.97      1751
     macro avg       0.93      0.95      0.94      1751
  weighted avg       0.97      0.97      0.97      1751



In [26]:
import os
import shutil
import json

# Save model locally in Colab
model.load_state_dict(best_model_state)
model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])

# Save training history
with open(f"{CONFIG['output_dir']}/training_history.json", 'w') as f:
    json.dump(training_history, f, indent=2)

print(f"Saved locally: {CONFIG['output_dir']}/")

# Save to Google Drive
drive_path = '/content/drive/MyDrive/immuniwatch_model'
if os.path.exists(drive_path):
    shutil.rmtree(drive_path)
shutil.copytree(CONFIG['output_dir'], drive_path)

print(f"Model saved to Google Drive at: MyDrive/immuniwatch_model")
print(f"Best Val F1: {best_val_f1:.4f}")
print("Save complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved locally: immuniwatch_model/
Model saved to Google Drive at: MyDrive/immuniwatch_model
Best Val F1: 0.9413
Save complete


In [28]:
# Download model as zip to your local machine
import shutil

zip_path = shutil.make_archive(
    base_name='immuniwatch_model',
    format='zip',
    root_dir=CONFIG['output_dir']
)

from google.colab import files
files.download('immuniwatch_model.zip')

print("Download started. Check your browser downloads.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started. Check your browser downloads.
